# V1 Policy Test — run the grid-trained ACT policy on the arm

Standalone deploy notebook for **models/act_v1** (229-episode systematic grid dataset).
`v0_test.ipynb` stays pointed at act_v0 for side-by-side comparison.

1. Run the setup cells (drivers → ROS libs → constants → node) — no SAM3/GraspGenX needed.
2. GO HOME, place the block, run DEPLOY. **Hand on the e-stop for first runs.**
3. First test: a block rotated 30–45° — the case V0 froze on.


In [1]:
# ── 1. Robot drivers: arm + gripper + camera + FT (NO SAM3/GraspGenX needed) ──
# Collect's cell 1 launches these as children of ITS kernel — restarting that kernel kills
# them. This cell brings up just the hardware drivers for the policy test (~15s, not 90).
import subprocess, os, time

ROS = 'source /opt/ros/humble/setup.bash && source ~/ws_ctrl/install/setup.bash'
def ros_proc(cmd, log):
    open(log, 'w').close()
    return subprocess.Popen(f'{ROS} && {cmd}', shell=True, executable='/bin/bash',
                            stdout=open(log, 'a'), stderr=subprocess.STDOUT)

print('Killing stale drivers...')
for p in ['ur5_node', 'gripper_node', 'realsense2_camera', 'ft_sensor_node']:
    subprocess.run(f'pkill -f {p}', shell=True)   # SIGTERM first: ur5_node releases RTDE cleanly
time.sleep(2.5)
for p in ['ur5_node', 'gripper_node', 'realsense2_camera', 'ft_sensor_node']:
    subprocess.run(f'pkill -9 -f {p}', shell=True)
time.sleep(3)

PROCS = {}
PROCS['ur5']     = ros_proc('ros2 run magpie_control ur5_node',       '/tmp/log_ur5.txt')
PROCS['gripper'] = ros_proc('ros2 run magpie_control gripper_node',   '/tmp/log_gripper.txt')
PROCS['ft']      = ros_proc('ros2 run magpie_control ft_sensor_node', '/tmp/log_ft.txt')
PROCS['camera']  = ros_proc(
    'ros2 run realsense2_camera realsense2_camera_node --ros-args -r __ns:=/camera/gripper_camera',
    '/tmp/log_camera.txt')
time.sleep(8)
for _try in range(2):   # ur5 auto-retry (dashboard-clear on start means a retry usually works)
    if PROCS['ur5'].poll() is None: break
    print(f'  ur5 EXITED — auto-retrying ({_try+1}/2)...')
    subprocess.run('pkill -9 -f ur5_node', shell=True); time.sleep(2)
    PROCS['ur5'] = ros_proc('ros2 run magpie_control ur5_node', '/tmp/log_ur5.txt')
    time.sleep(6)
print('drivers:', {k: ('RUNNING' if v.poll() is None else 'EXITED!') for k, v in PROCS.items()})
print('If any say EXITED!, check its /tmp/log_*.txt')


Killing stale drivers...
drivers: {'ur5': 'RUNNING', 'gripper': 'RUNNING', 'ft': 'EXITED!', 'camera': 'RUNNING'}
If any say EXITED!, check its /tmp/log_*.txt


In [2]:
# Load ROS shared libraries so rclpy imports in any VS Code kernel
import sys, os, ctypes, glob

for _d in [
    '/opt/ros/humble/lib/x86_64-linux-gnu',
    '/opt/ros/humble/lib',
    '/home/user/ws_ctrl/install/magpie_msgs/lib',
    '/home/user/ws_ctrl/install/magpie_control/lib',
]:
    for _so in sorted(glob.glob(_d + '/*.so*')):
        try:
            ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass

for _p in [
    '/opt/ros/humble/local/lib/python3.10/dist-packages',
    '/opt/ros/humble/lib/python3.10/site-packages',
    '/home/user/ws_ctrl/install/magpie_msgs/local/lib/python3.10/dist-packages',
    '/home/user/ws_ctrl/install/magpie_control/lib/python3.10/site-packages',
    '/home/user/.local/lib/python3.10/site-packages',
    '/home/user/magpie_control/src',
    '/home/user/magpie_control/scripts',
]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import rclpy
print('rclpy OK')

         at line 253 in /opt/ros/humble/include/class_loader/class_loader/class_loader_core.hpp
         at line 253 in /opt/ros/humble/include/class_loader/class_loader/class_loader_core.hpp


rclpy OK


In [3]:
# scripts/ on the path (collect's cell 1 normally does this; v0_test is standalone)
import sys as _sys, os as _os
_sp = _os.path.expanduser('~/magpie_control/scripts')
if _sp not in _sys.path: _sys.path.insert(0, _sp)

import time, re, json, tempfile, base64, socket as _sock
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
import cv2
%matplotlib inline
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['figure.dpi']     = 80   # keeps output file size small for GitHub

from rclpy.node import Node
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy
from sensor_msgs.msg import Image as RosImage, CameraInfo
from geometry_msgs.msg import PoseStamped, Pose, WrenchStamped
from std_srvs.srv import Trigger
from cv_bridge import CvBridge
from magpie_msgs.msg import GripperState
from magpie_msgs.srv import MoveLinear, SetGripperForce, SetGripperPosition
from magpie_control import poses
from magpie_control.homog_utils import homog_xform, R_krot
from magpie_control.gripper_arc import fingertip_drop
from google import genai
from google.genai import types as gtypes
from pointcloud_utils import build_segmented_pcd, denoise_pcd, analyse_pcd

# --- CONFIG ---
OBJECT         = ''   # blank = Gemini auto-detects
SAM3_SOCK      = '/tmp/sam3.sock'
GEMINI_KEY     = os.environ.get('GEMINI_API_KEY', '')
GRIPPER_LEN    = 0.231
HARD_FLOOR_Z   = 0.060   # overridden by TABLE_Z cell → always tracks real table surface
TABLE_Z        = None   # set by running the measure cell below
CAMERA_MOUNT_Z_OFFSET_M = 0.024  # extend OBB aug bottom below table surface; z_off in _TCP_TO_CAM already accounts for mount height
APPROACH_H     = 0.10
# Camera extrinsic: transform from camera optical frame to TCP frame.
# z_off=0.144 m = physical camera-to-TCP distance (0.120 + 0.024 measured mount error).
# Rz(-90°) is the last-working clocking — run the calibration cell (cell 7) at setup
# to re-measure the correct clocking and confirm z_off for this mount.
# The calibration cell overwrites _TCP_TO_CAM for the session.
# XY offset of camera center relative to gripper jaw center in tool frame (metres).
# Tune by observing consistent grasp miss direction: if arm always misses +5mm in world-X,
# add that error (in tool frame) here.  Positive X = camera is offset toward robot.
_CAM_XY_OFFSET_M = (0.0, 0.0)  # (x_tool_m, y_tool_m) — adjust to compensate lateral camera offset
_TCP_TO_CAM    = homog_xform(R_krot([0, 0, 1], -np.pi/2), [_CAM_XY_OFFSET_M[0], _CAM_XY_OFFSET_M[1], 0.144])  # z_off updated: 0.120+0.024 mount

# sam3_query defined HERE (early) so every downstream cell has it after a kernel reload
# — no more 'sam3_query is not defined' from running cells out of order.
def sam3_query(img_rgb, query, sock_path=SAM3_SOCK):
    tmp = tempfile.mktemp(suffix='.jpg')
    cv2.imwrite(tmp, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    try:
        with _sock.socket(_sock.AF_UNIX, _sock.SOCK_STREAM) as s:
            s.connect(sock_path)
            s.sendall((json.dumps({'image': tmp, 'query': query}) + '\n').encode())
            raw = b''
            while True:
                chunk = s.recv(65536)
                if not chunk: break
                raw += chunk
        d = json.loads(raw.decode().strip())
        if 'error' in d: raise RuntimeError(d['error'])
        boxes  = np.array(d['boxes'],  dtype=float)
        scores = np.array(d['scores'], dtype=float)
        mask   = None
        if d.get('mask_b64') and len(boxes) > 0:
            raw2 = base64.b64decode(d['mask_b64'])
            h, w = d['mask_shape']
            mask = np.frombuffer(raw2, dtype=np.uint8).reshape(h, w).astype(bool)
        return boxes, scores, mask
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

if not GEMINI_KEY:
    print('WARNING: GEMINI_API_KEY not set.')
print('Imports OK')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Imports OK


In [4]:
# Helper: quaternion <-> rotation vector
def _quat_to_rv(w, x, y, z):
    a = 2.0 * np.arccos(np.clip(w, -1, 1))
    s = np.sin(a / 2)
    return np.zeros(3) if s < 1e-10 else a * np.array([x, y, z]) / s

def _rv_to_quat(rv):
    a = np.linalg.norm(rv)
    if a < 1e-10:
        return (1., 0., 0., 0.)
    ax = rv / a
    return (np.cos(a/2), ax[0]*np.sin(a/2), ax[1]*np.sin(a/2), ax[2]*np.sin(a/2))

def _mat_to_pose(mat):
    v = poses.pose_mtrx_to_vec(np.array(mat))
    w, x, y, z = _rv_to_quat(np.array(v[3:]))
    p = Pose()
    p.position.x, p.position.y, p.position.z = v[0], v[1], v[2]
    p.orientation.w, p.orientation.x = w, x
    p.orientation.y, p.orientation.z = y, z
    return p

# QoS profiles matching RealSense D405 publisher settings
img_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                     durability=DurabilityPolicy.TRANSIENT_LOCAL)
inf_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                     durability=DurabilityPolicy.VOLATILE)

class Demo(Node):
    def __init__(self):
        super().__init__('magpie_demo')
        self.bridge = CvBridge()
        self.color = self.depth = self.caminfo = self.tcp = self.gs = None
        self.wrench = None   # WrenchStamped from OptoForce FT (may be None)
        NS = '/camera/gripper_camera/camera'
        self.create_subscription(
            RosImage,    NS+'/color/image_raw',
            lambda m: setattr(self, 'color', self.bridge.imgmsg_to_cv2(m, 'rgb8')), img_qos)
        self.create_subscription(
            RosImage,    NS+'/depth/image_rect_raw',
            lambda m: setattr(self, 'depth', self.bridge.imgmsg_to_cv2(m, 'passthrough')), img_qos)
        self.create_subscription(
            CameraInfo,  NS+'/color/camera_info',
            lambda m: setattr(self, 'caminfo', m), inf_qos)
        self.create_subscription(
            GripperState, '/gripper/state',
            lambda m: setattr(self, 'gs', m), 1)
        self.create_subscription(
            WrenchStamped, 'ft_sensor/wrench',
            lambda m: setattr(self, 'wrench', m), 10)
        self.create_subscription(
            PoseStamped, '/arm/tcp_pose',
            self._tcp_cb, 1)
        self.mv  = self.create_client(MoveLinear,         '/arm/move_l')
        self.tch = self.create_client(Trigger,             '/arm/teach_mode')
        self.opn = self.create_client(Trigger,             '/gripper/open')
        self.cls = self.create_client(Trigger,             '/gripper/close')
        self.frc = self.create_client(SetGripperForce,     '/gripper/set_force')
        self.pos = self.create_client(SetGripperPosition,  '/gripper/set_position')
        self.clr = self.create_client(Trigger,             '/gripper/clear_error')

    def _tcp_cb(self, m):
        p = m.pose
        rv = _quat_to_rv(p.orientation.w, p.orientation.x, p.orientation.y, p.orientation.z)
        self.tcp = poses.pose_vec_to_mtrx([p.position.x, p.position.y, p.position.z, *rv])

    def spin(self, n=10, t=0.15):
        for _ in range(n):
            rclpy.spin_once(self, timeout_sec=t)

    def _call(self, client, req, timeout=30.):
        client.wait_for_service(timeout_sec=4.)
        fut = client.call_async(req)
        rclpy.spin_until_future_complete(self, fut, timeout_sec=timeout)
        return fut.result()

    def move(self, mat, spd=0.08, acc=0.2):
        r = MoveLinear.Request()
        r.target_pose = _mat_to_pose(mat)
        r.speed = spd; r.acceleration = acc; r.async_mode = False
        resp = self._call(self.mv, r)
        if not resp.success:
            raise RuntimeError(resp.message)

    def open_g(self):  return self._call(self.opn, Trigger.Request())
    def close_g(self): return self._call(self.cls, Trigger.Request())

    def set_force(self, n):
        r = SetGripperForce.Request(); r.max_force = float(n)
        return self._call(self.frc, r)

    def set_pos(self, mm):
        r = SetGripperPosition.Request(); r.position = float(max(0, mm))
        return self._call(self.pos, r)

    def clear_err(self):
        # Re-enable AX-12 torque after overload shutdown (reset_packet_overload).
        # NOTE: this resets force limit to 2N, so call set_force() again after.
        if not self.clr.wait_for_service(timeout_sec=1.):
            return None
        return self._call(self.clr, Trigger.Request())

    def slip_guard_enable(self, force_n, slip_thresh, obj_u=None, obj_v=None, force_step=1.0):
        """Send config (force, thresh, object pixel coords, reclamp step) then enable the SlipGuardNode."""
        if not hasattr(self, '_sg_en'):
            from std_msgs.msg import Float32MultiArray
            self._sg_cfg_pub = self.create_publisher(
                Float32MultiArray, 'slip_guard/config', 1)
            self._sg_en  = self.create_client(Trigger, 'slip_guard/enable')
            self._sg_dis = self.create_client(Trigger, 'slip_guard/disable')
            from std_msgs.msg import String as _SgStr
            self._sg_events = []
            self.create_subscription(_SgStr, '/slip_guard/events',
                lambda msg: self._sg_events.append(msg.data), 10)
        from std_msgs.msg import Float32MultiArray
        cfg = Float32MultiArray()
        # [force, thresh, obj_u, obj_v] — pixel coords tell guard where to sample depth
        cfg.data = [float(force_n), float(slip_thresh),
                    float(obj_u) if obj_u is not None else -1.,
                    float(obj_v) if obj_v is not None else -1.,
                    float(force_step)]
        self._sg_events = []  # reset for this grasp — fresh DAgger log
        self._sg_cfg_pub.publish(cfg)
        time.sleep(0.1)
        if not self._sg_en.wait_for_service(timeout_sec=1.):
            print('[slip_guard] node not running — guard skipped')
            return
        self._call(self._sg_en, Trigger.Request())

    def slip_guard_disable(self):
        if hasattr(self, '_sg_dis') and self._sg_dis.wait_for_service(timeout_sec=0.5):
            self._call(self._sg_dis, Trigger.Request())

    def unteach(self):
        if not self.tch.wait_for_service(timeout_sec=2.):
            return
        r = self._call(self.tch, Trigger.Request())
        if r and 'enabled' in r.message.lower():
            self._call(self.tch, Trigger.Request())

    def wait_sensors(self, timeout=20.):
        t0 = time.time()
        while time.time() - t0 < timeout:
            rclpy.spin_once(self, timeout_sec=0.15)
            if all(v is not None for v in [self.color, self.depth, self.caminfo, self.tcp]):
                return True
        return False

try:
    rclpy.init()
except RuntimeError:
    pass

try:
    node.destroy_node()
except Exception:
    pass

node = Demo()
ok   = node.wait_sensors(20.)
print('Sensors ready:', ok,
      '| color:', node.color is not None,
      '| depth:', node.depth is not None,
      '| tcp:',   node.tcp   is not None)

Sensors ready: True | color: True | depth: True | tcp: True


In [5]:
# ── GO HOME + GRIPPER RESET — run before each policy rollout ─────────────────
# HOME = the median frame-0 pose of all 60 training episodes (the exact overhead start
# the policy expects; orientation consistent to ±0.1° across the dataset).
# Two-stage move: vertical to home height first, then lateral — no diagonal swoops.
import numpy as np, time

HOME_VEC = [-0.1696, -0.5568, 0.4071, -3.1184, 0.0469, -0.0387]   # from the dataset itself

# Release servo mode first — after servoL streaming (the deploy loop) the controller stays
# in servo state and REFUSES moveL ('unreachable or singular') until servoStop is called.
from std_srvs.srv import Trigger as _TrigS
if not hasattr(node, '_arm_stop_cli'):
    node._arm_stop_cli = node.create_client(_TrigS, '/arm/stop')
if node._arm_stop_cli.wait_for_service(timeout_sec=3.):
    _sr = node._call(node._arm_stop_cli, _TrigS.Request())
    print(f'  [home] servo released ({getattr(_sr, "message", "?")})')
node.unteach(); node.clear_err(); node.spin(4)
_home = poses.pose_vec_to_mtrx(HOME_VEC)
_st1 = node.tcp.copy(); _st1[2, 3] = _home[2, 3]      # stage 1: match home height
node.move(_st1, spd=0.08); time.sleep(0.3)
node.move(_home, spd=0.08); time.sleep(0.3); node.spin(4)   # stage 2: exact home pose
_err = np.linalg.norm(node.tcp[:3, 3] - _home[:3, 3]) * 1000.
print(f'  [home] at collection home ({_err:.0f}mm from target)')

# gripper reset: re-enable torque, recalibrate fingers, re-assert force, open
from std_srvs.srv import Trigger as _TrigG
node.clear_err()
if not hasattr(node, '_grip_cal_cli'):
    node._grip_cal_cli = node.create_client(_TrigG, '/gripper/calibrate')
try:
    if node._grip_cal_cli.wait_for_service(timeout_sec=3.):
        _r = node._call(node._grip_cal_cli, _TrigG.Request())
        print(f'  [gripper] calibrate -> {getattr(_r, "message", "no response")}')
    else:
        print('  [gripper] calibrate service unavailable — clear_err + open only')
except Exception as _ge:
    print(f'  [gripper] calibrate skipped ({_ge})')
node.clear_err(); node.set_force(16.); node.open_g()
print('  [ready] arm at home, gripper open — place the block, then run DEPLOY')


  [home] servo released (Arm stopped)
  [home] at collection home (0mm from target)
  [gripper] calibrate -> Gripper calibrated successfully
  [ready] arm at home, gripper open — place the block, then run DEPLOY


In [6]:
# ═══ DEPLOY V1 — trained ACT policy on the arm (verified offline) ═══════════
# Offline replay vs training data: position 1.9mm mean | rotation p95 1.7deg | grip exact.
# Design (each choice validated against a failure we hit):
#   * n_action_steps=10, NO temporal ensembling — ensembling AVERAGES rotation vectors
#     across chunks, which is invalid near 180 deg (made rotation 3x worse in replay).
#     Small chunks = re-infer every 1s with fresh observations, actions stay coherent.
#   * rotvec canonicalized to the branch nearest the current pose (kills 180-flip lurches)
#   * rotation-RATE limit 12 deg/tick + TILT gate 25 deg (wrist yaw FREE — needed to align
#     to the block's flat sides; the old total-angle cap caused the edge grasps)
#   * EMA position smoothing + 15mm/tick step cap + XY box + Z floor incl. GRIPPER_LEN
#   * driver-side: _SERVO_TIME now 0.1s (was 2ms bursts + 98ms freeze = mechanical stutter)
import sys, os, time, types
import numpy as np, cv2

POLICY_DIR   = '/home/user/magpie_control/models/act_v1'   # the 229-episode grid policy
TASK         = 'pick up the red block'
DEPLOY_STEPS = 600          # ~60s at 10Hz — off-centre reaches creep slowly (OOD hesitance)
BOX_HALF_XY  = 0.10
Z_MIN        = None         # default: table + GRIPPER_LEN + 3mm
Z_MAX        = 0.45
STEP_CAP_M   = 0.015
EMA_ALPHA    = 0.4
TILT_CAP_DEG = 25.          # tool tilt from start orientation; yaw unrestricted
ROT_RATE_CAP = 12.          # max orientation change per tick (deg)
GRIP_FORCE_N = 16.          # matches the force the demos closed at

# ── load policy (cached; delete _dep_policy_v1a to force reload) ───────────────
if '_dep_policy_v1a' not in dir():
    for _m in ['lerobot.policies.groot', 'lerobot.policies.groot.configuration_groot',
               'lerobot.policies.groot.modeling_groot']:
        if _m not in sys.modules: sys.modules[_m] = types.ModuleType(_m)
    sys.modules['lerobot.policies.groot.configuration_groot'].GrootConfig = type('GrootConfig', (), {})
    sys.modules['lerobot.policies.groot.modeling_groot'].GrootPolicy = type('GrootPolicy', (), {})
    import torch as _th
    from lerobot.policies.act.modeling_act import ACTPolicy
    from lerobot.configs.policies import PreTrainedConfig
    from lerobot.policies.factory import make_pre_post_processors
    from lerobot.utils.control_utils import predict_action as _predict_action
    _cfg = PreTrainedConfig.from_pretrained(POLICY_DIR)
    _cfg.n_action_steps = 10          # re-infer every 1s; raw coherent chunks (no ensembling)
    _dep_device = _th.device('cuda' if _th.cuda.is_available() else 'cpu')
    try:
        _dep_policy_v1a = ACTPolicy.from_pretrained(POLICY_DIR, config=_cfg).to(_dep_device).eval()
    except RuntimeError:
        _dep_device = _th.device('cpu')
        _dep_policy_v1a = ACTPolicy.from_pretrained(POLICY_DIR, config=_cfg).to(_dep_device).eval()
    _dep_pre, _dep_post = make_pre_post_processors(
        policy_cfg=_dep_policy_v1a.config, pretrained_path=POLICY_DIR,
        preprocessor_overrides={'device_processor': {'device': str(_dep_device)}})
    print(f'  [deploy] policy on {_dep_device}, n_action_steps=10 (1s recompute, no ensembling)')
import torch
from geometry_msgs.msg import PoseStamped as _DepPS
if not hasattr(node, '_servo_pub'):
    node._servo_pub = node.create_publisher(_DepPS, '/arm/servo_l_cmd', 10)

# ── start state + safety envelope ─────────────────────────────────────────────
# release any stale servo state from a previous/interrupted run BEFORE starting
from std_srvs.srv import Trigger as _TrigS0
if not hasattr(node, '_arm_stop_cli'):
    node._arm_stop_cli = node.create_client(_TrigS0, '/arm/stop')
if node._arm_stop_cli.wait_for_service(timeout_sec=3.):
    node._call(node._arm_stop_cli, _TrigS0.Request())
node.unteach(); node.clear_err(); node.spin(6)
_start_tcp = node.tcp.copy()
_R_last_ok = _start_tcp[:3, :3].copy()
_z_ax0     = _start_tcp[:3, 2].copy()
_box_c     = _start_tcp[:2, 3].copy()
_tbl   = TABLE_Z if ('TABLE_Z' in dir() and TABLE_Z is not None) else 0.035
_z_min = Z_MIN if Z_MIN is not None else (_tbl + GRIPPER_LEN + 0.003)
_sm_p  = _start_tcp[:3, 3].copy()
print(f'  [deploy] box ({_box_c[0]:+.3f},{_box_c[1]:+.3f})±{BOX_HALF_XY*100:.0f}cm  '
      f'z∈[{_z_min:.3f},{Z_MAX:.3f}]  step≤{STEP_CAP_M*1000:.0f}mm  '
      f'tilt≤{TILT_CAP_DEG:.0f}° rot-rate≤{ROT_RATE_CAP:.0f}°/tick  yaw FREE')
node.set_force(GRIP_FORCE_N); node.open_g(); _grip_closed = False
_preposed = False; _low_ticks = 0    # aperture pre-position trigger (see below)
_dep_policy_v1a.reset()

# ── 10Hz control loop ─────────────────────────────────────────────────────────
print(f'  [deploy] running "{TASK}" for {DEPLOY_STEPS} steps — hand on the e-stop!')
_dt = 0.1; _clamped_n = 0; _inf_ms = []
for _step in range(DEPLOY_STEPS):
    _t0 = time.time()
    node.spin(1)
    if node.color is None or node.tcp is None:
        print('  [deploy] no camera/tcp — stopping'); break
    _img = np.ascontiguousarray(node.color, dtype=np.uint8)
    _tv  = poses.pose_mtrx_to_vec(np.array(node.tcp))
    _gs  = node.gs
    _wz  = float(node.wrench.wrench.force.z) if getattr(node, 'wrench', None) is not None else 0.
    _state = np.array([*_tv, (_gs.position if _gs else 0.), (_gs.force if _gs else 0.), _wz],
                      dtype=np.float32)
    _ti = time.time()
    _a = _predict_action({'observation.images.wrist': _img, 'observation.state': _state},
                         _dep_policy_v1a, _dep_device, _dep_pre, _dep_post,
                         use_amp=False, task=TASK, robot_type='ur5_magpie')
    _inf_ms.append((time.time()-_ti)*1000.)
    _a = np.asarray(_a, dtype=float).ravel()
    _tgt_p, _tgt_rv, _grip = _a[:3].copy(), _a[3:6].copy(), float(_a[6])

    # rotvec branch canonicalization (same rotation, branch nearest the current pose)
    _cur_rv = np.array(_tv[3:6])
    _nrv = float(np.linalg.norm(_tgt_rv))
    if _nrv > 1e-6:
        _alt = _tgt_rv * (1. - 2.*np.pi/_nrv)
        if np.linalg.norm(_alt - _cur_rv) < np.linalg.norm(_tgt_rv - _cur_rv):
            _tgt_rv = _alt
    _cl = False
    # position: box + EMA + step cap
    _tgt_p[0] = np.clip(_tgt_p[0], _box_c[0]-BOX_HALF_XY, _box_c[0]+BOX_HALF_XY)
    _tgt_p[1] = np.clip(_tgt_p[1], _box_c[1]-BOX_HALF_XY, _box_c[1]+BOX_HALF_XY)
    _tgt_p[2] = np.clip(_tgt_p[2], _z_min, Z_MAX)
    _sm_p = (1.-EMA_ALPHA)*_sm_p + EMA_ALPHA*_tgt_p
    _cur_p = np.array(_tv[:3])
    _dvec = _sm_p - _cur_p; _dn = float(np.linalg.norm(_dvec))
    _cmd_p = _cur_p + _dvec/_dn*STEP_CAP_M if _dn > STEP_CAP_M else _sm_p
    if _dn > STEP_CAP_M: _cl = True
    # orientation: rate-limit toward the commanded rotation, then tilt-gate (yaw free)
    _R_cmd = poses.pose_vec_to_mtrx([*_cmd_p, *_tgt_rv])[:3, :3]
    _rv_rel = cv2.Rodrigues(_R_last_ok.T @ _R_cmd)[0].ravel()
    _ang_rel = float(np.degrees(np.linalg.norm(_rv_rel)))
    if _ang_rel > ROT_RATE_CAP:
        _R_cmd = _R_last_ok @ cv2.Rodrigues(_rv_rel * (ROT_RATE_CAP/_ang_rel))[0]; _cl = True
    _tilt = float(np.degrees(np.arccos(np.clip(np.dot(_R_cmd[:, 2], _z_ax0), -1., 1.))))
    if _tilt > TILT_CAP_DEG:
        _R_cmd = _R_last_ok; _cl = True
    else:
        _R_last_ok = _R_cmd.copy()
    if _cl: _clamped_n += 1
    _tgt_mat = np.eye(4); _tgt_mat[:3, :3] = _R_cmd; _tgt_mat[:3, 3] = _cmd_p

    _ps = _DepPS()
    _ps.pose = _mat_to_pose(_tgt_mat)
    node._servo_pub.publish(_ps)

    if _grip > 0.5 and not _grip_closed:
        print(f'  [deploy] step {_step}: policy CLOSES gripper')
        # ASYNC chain (clear_err -> re-assert force -> close): blocking service calls froze the
        # servo stream ~1s each and the controller watchdog KILLED the RTDE script mid-run.
        node.clr.call_async(_TrigS0.Request())
        _fr = SetGripperForce.Request(); _fr.max_force = float(GRIP_FORCE_N)
        node.frc.call_async(_fr)
        node.cls.call_async(_TrigS0.Request())
        _grip_closed = True; _close_step = _step
    elif _grip < 0.5 and _grip_closed:
        print(f'  [deploy] step {_step}: policy OPENS gripper')
        node.clr.call_async(_TrigS0.Request()); node.opn.call_async(_TrigS0.Request())
        _grip_closed = False
    # ── aperture pre-position shim ── in EVERY demo, the script pre-positioned the fingers
    # to ~obj_width+5mm (ap 103.6 -> ~53) once at grasp height, and the policy's close fires
    # 0.5-3s AFTER that snap. The pre-position is NOT in the action space (grip is binary),
    # so deploy must reproduce it exogenously, exactly like the training environment did.
    if not _preposed and not _grip_closed:
        _low_ticks = _low_ticks + 1 if _tv[2] < 0.325 else 0
        if _low_ticks >= 10:                     # ~1s stable at grasp height
            print(f'  [deploy] step {_step}: at grasp height — pre-positioning fingers to 51mm (as demos did)')
            _pr = SetGripperPosition.Request(); _pr.position = 51.
            node.pos.call_async(_pr); _preposed = True

    if _step % 20 == 0:
        _Ry = cv2.Rodrigues(np.array(_tv[3:6]))[0]
        _yaw_now = float(np.degrees(np.arctan2(_Ry[1,0], _Ry[0,0])))
        print(f'  [deploy] step {_step:4d}  pos=({_tv[0]:+.3f},{_tv[1]:+.3f},{_tv[2]:+.3f})  '
              f'yaw={_yaw_now:+4.0f}°  ap={(_gs.position if _gs else 0):5.1f}mm  '
              f'tgt_z={_a[2]:+.3f}  pred_grip={_grip:+.2f}  {"CLOSED" if _grip_closed else "open"}  '
              f'clamped={_clamped_n}')
    # 3s after the policy closes, hand off to the scripted place (same division of labour
    # as the demos: policy = perceive/reach/align/close, script = lift-carry-place)
    if _grip_closed and '_close_step' in dir() and (_step - _close_step) >= 35:
        _ap_now = float(_gs.position) if _gs else 0.
        if _ap_now < 15.:   # fingers closed on AIR — a miss, nothing to place
            print(f'  [deploy] step {_step}: MISS — closed on air (ap={_ap_now:.1f}mm). Opening, no place.')
            node.clr.call_async(_TrigS0.Request()); node.opn.call_async(_TrigS0.Request())
            _grip_closed = False
            break
        print(f'  [deploy] step {_step}: grasp held (ap={_ap_now:.1f}mm) — handing off to scripted place')
        break
    _sl = _dt - (time.time() - _t0)
    if _sl > 0: time.sleep(_sl)

# ── scripted place: lift → carry to centre → lower → release → rise ──────────
if _grip_closed:
    time.sleep(1.0); node.spin(6)                     # let the close finish + state settle
    _ap_fin = float(node.gs.position) if node.gs else 0.
    if _ap_fin < 15.:
        print(f'  [deploy] MISS — closed on air (ap={_ap_fin:.1f}mm). Opening, skipping place.')
        node.clr.call_async(_TrigS0.Request()); node.opn.call_async(_TrigS0.Request())
        _grip_closed = False
if _grip_closed:
    try:
        if node._arm_stop_cli.wait_for_service(timeout_sec=3.):
            node._call(node._arm_stop_cli, _TrigS0.Request())   # leave servo mode for moveL
        node.unteach(); node.spin(3)
        _pl = node.tcp.copy(); _pl[2, 3] = 0.407
        node.move(_pl, spd=0.08); time.sleep(0.3)               # lift straight up (holding)
        _pl[0, 3], _pl[1, 3] = _box_c[0], _box_c[1]
        node.move(_pl, spd=0.08); time.sleep(0.3)               # carry to the start centre
        _pl[2, 3] = 0.318                                        # demo place height (+8mm)
        node.move(_pl, spd=0.05); time.sleep(0.3)               # lower to just above table
        node.clear_err(); node.open_g(); time.sleep(0.8)        # release
        _grip_closed = False
        _pl[2, 3] = 0.407
        node.move(_pl, spd=0.08)                                 # rise clear
        print('  [deploy] placed at centre and released ✓')
    except Exception as _pe:
        print(f'  [deploy] scripted place failed ({_pe})')
        print('           if it says unreachable/singular: the RTDE script died — re-run cell 1 (drivers).')
        print('           the block may still be in the gripper: node.open_g() once the arm is low.')
# release servo mode so subsequent moveL commands (GO HOME etc.) are accepted
try:
    from std_srvs.srv import Trigger as _TrigS2
    if not hasattr(node, '_arm_stop_cli'):
        node._arm_stop_cli = node.create_client(_TrigS2, '/arm/stop')
    if node._arm_stop_cli.wait_for_service(timeout_sec=3.):
        node._call(node._arm_stop_cli, _TrigS2.Request())
        print('  [deploy] servo mode released')
except Exception as _se:
    print(f'  [deploy] servo release skipped ({_se})')
print(f'\n  [deploy] done: {min(_step+1, DEPLOY_STEPS)} steps, {_clamped_n} clamped, '
      f'inference {np.mean(_inf_ms) if _inf_ms else 0:.0f}ms/tick. '
      f'Gripper {"still CLOSED — node.open_g() to release" if _grip_closed else "open"}.')


Loading weights from local directory
  [deploy] policy on cuda, n_action_steps=10 (1s recompute, no ensembling)
  [deploy] box (-0.170,-0.557)±10cm  z∈[0.269,0.450]  step≤15mm  tilt≤25° rot-rate≤12°/tick  yaw FREE
  [deploy] running "pick up the red block" for 600 steps — hand on the e-stop!
  [deploy] step    0  pos=(-0.170,-0.557,+0.407)  yaw=  -2°  ap=103.6mm  tgt_z=+0.322  pred_grip=-0.00  open  clamped=1
  [deploy] step   20  pos=(-0.170,-0.556,+0.362)  yaw=  -2°  ap=103.6mm  tgt_z=+0.318  pred_grip=+0.00  open  clamped=21
  [deploy] step   40  pos=(-0.170,-0.557,+0.317)  yaw=  -2°  ap=103.6mm  tgt_z=+0.317  pred_grip=+0.00  open  clamped=28
  [deploy] step 41: at grasp height — pre-positioning fingers to 51mm (as demos did)
  [deploy] step   60  pos=(-0.170,-0.557,+0.317)  yaw=  -2°  ap=103.6mm  tgt_z=+0.318  pred_grip=+0.00  open  clamped=28
  [deploy] step   80  pos=(-0.170,-0.557,+0.317)  yaw=  -2°  ap= 51.0mm  tgt_z=+0.318  pred_grip=+0.00  open  clamped=28
  [deploy] step  1